# Sup. Fig. 7: runtime

Every method of `06_performance` re-timed three times on this server across the gene set sizes of Sup. Figs. 5-6:
EnrichMap with the null model (default since 0.3.0) and with smoothing only, AUCell, GSVA, ssGSEA, Z score, Scanpy and
GSDensity (R, run in its own environment; timings read from the cache). Panels: 7a-b runtime against gene set size,
compact and full, and a runtime summary. Source data are saved as CSV next to each figure.

In [1]:
import os
import warnings

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

In [2]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np
import enrichmap as em
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pickle

In [3]:
sc.set_figure_params(frameon=False, fontsize=6)
sc.settings._vector_friendly = False
plt.rcParams["axes.grid"] = False

In [4]:
# set working directory
# repository root (the folder containing `notebooks/`); safe to re-run this cell
import sys
project_dir = os.getcwd()
if os.path.basename(project_dir) == "notebooks":
    project_dir = os.path.dirname(project_dir)
project_dir += "/"
sys.path.insert(0, project_dir + "notebooks")  # helpers.py and score_signatures.py
working_dir = project_dir
os.chdir(working_dir)

# set figure directory
figure_dir = working_dir + "figures/"
os.makedirs(figure_dir, exist_ok=True)

# processed data directory
processed_data = project_dir + "processed_data/"

# source data directory
source_dir = working_dir + "source_data/"
os.makedirs(source_dir, exist_ok=True)

In [5]:
from helpers import (
    set_box_ratio,
    runtime_method_legend,
    runtime_plot_summary,
    runtime_time_method,
    runtime_vs_size_axis,
)

In [6]:
colours = {
    "EnrichMap (null)": "#2f0bdf",
    "AUCell": "#86bf6e",
    "GSVA": "#c76663",
    "ssGSEA": "#8f6eb0",
    "Z score": "#866342",
    "Scanpy": "#cf83bd",
    "GSDensity": "#e3a33b",
}
spot_size = 20

In [7]:
# 1. Load data and gene sets (06_performance: no gene filter)
adata = sq.datasets.visium_hne_adata()
adata.layers["counts"] = adata.raw.X.copy()
adata.X = adata.layers["counts"].copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["log1p_norm"] = adata.X.copy()

with open(project_dir + "Pyramidal_layer_signatures.pkl", "rb") as f:
    pyramidal_dict = pickle.load(f)

with open(project_dir + "Striatum_signatures.pkl", "rb") as f:
    striatum_dict = pickle.load(f)

cell_type_1 = "Pyramidal_layer"
cell_type_2 = "Striatum"
regions = [cell_type_1, cell_type_2]
gene_sets = {
    cell_type_1: {len(v): list(v) for v in pyramidal_dict.values()},
    cell_type_2: {len(v): list(v) for v in striatum_dict.values()},
}
sizes = {ct: list(gene_sets[ct]) for ct in regions}

# prepare gene expression DataFrame (outside the timing, as in 06_performance)
X_dense = adata.X.toarray() if not isinstance(adata.X, np.ndarray) else adata.X
gene_expr = pd.DataFrame(X_dense.T, index=adata.var_names, columns=adata.obs_names)
gene_expr.index = gene_expr.index.str.upper()
del X_dense

In [8]:
# 2. One timed call per method (06_performance timing loop; gseapy with 2 threads)
variants = {
    "null": dict(null_model=True, correct_spatial_covariates=False),
    "smooth": dict(null_model=False, correct_spatial_covariates=False),
}
enrichmap_variants = {"EnrichMap (null)": "null"}
methods = list(colours)
python_methods = ["EnrichMap (null)", "AUCell", "Scanpy", "ssGSEA", "Z score"]
n_repeats = 3
threads = 2


In [9]:
# 3. Warm-up: one untimed call per Python method (imports, numba JIT, gseapy start-up)
warm_up = pd.DataFrame([
    {"method": method, "first call (s), Pyramidal_layer n = 10": runtime_time_method(
        adata, gene_expr, method, cell_type_1, gene_sets[cell_type_1][10], variants, enrichmap_variants, threads
    )}
    for method in python_methods
])
print(warm_up.round(3).to_string(index=False))

Scoring Pyramidal_layer_null: 10/10 genes found: 100%|██████████| 1/1 [00:00<00:00, 11.52it/s]


          method  first call (s), Pyramidal_layer n = 10
EnrichMap (null)                                   1.886
          AUCell                                  47.055
          Scanpy                                   0.613
          ssGSEA                                  23.159
         Z score                                   0.023


In [10]:
# 4. Timing: three interleaved repeats (repeat -> region -> size -> method) of every Python method
rows = []
for rep in range(1, n_repeats + 1):
    for ct in regions:
        for n in sizes[ct]:
            for method in python_methods:
                rows.append({
                    "repeat": rep,
                    "region": ct,
                    "method": method,
                    "gene set size": n,
                    "time (s)": runtime_time_method(adata, gene_expr, method, ct, gene_sets[ct][n], variants,
                                                 enrichmap_variants, threads),
                    "load average (1 min)": os.getloadavg()[0],
                })
    print(f"repeat {rep} done", flush=True)
timings_py = pd.DataFrame(rows)
print(timings_py.pivot_table(index=["region", "gene set size"], columns="method", values="time (s)", aggfunc="mean")
      [python_methods].round(3).to_string())

Scoring Pyramidal_layer_null: 50/50 genes found: 100%|██████████| 1/1 [00:00<00:00, 11.24it/s]
Scoring Pyramidal_layer_null: 100/100 genes found: 100%|██████████| 1/1 [00:00<00:00, 10.85it/s]
Scoring Pyramidal_layer_null: 200/200 genes found: 100%|██████████| 1/1 [00:00<00:00, 11.22it/s]
Scoring Pyramidal_layer_null: 500/500 genes found: 100%|██████████| 1/1 [00:00<00:00,  9.54it/s]
Scoring Striatum_null: 434/434 genes found: 100%|██████████| 1/1 [00:00<00:00, 10.68it/s]


repeat 1 done


Scoring Pyramidal_layer_null: 50/50 genes found: 100%|██████████| 1/1 [00:00<00:00, 13.94it/s]
Scoring Pyramidal_layer_null: 100/100 genes found: 100%|██████████| 1/1 [00:00<00:00, 13.70it/s]
Scoring Pyramidal_layer_null: 200/200 genes found: 100%|██████████| 1/1 [00:00<00:00, 13.84it/s]
Scoring Pyramidal_layer_null: 500/500 genes found: 100%|██████████| 1/1 [00:00<00:00, 10.60it/s]
Scoring Striatum_null: 434/434 genes found: 100%|██████████| 1/1 [00:00<00:00, 12.22it/s]


repeat 2 done


Scoring Pyramidal_layer_null: 50/50 genes found: 100%|██████████| 1/1 [00:00<00:00, 15.90it/s]
Scoring Pyramidal_layer_null: 100/100 genes found: 100%|██████████| 1/1 [00:00<00:00, 14.77it/s]
Scoring Pyramidal_layer_null: 200/200 genes found: 100%|██████████| 1/1 [00:00<00:00, 11.03it/s]
Scoring Pyramidal_layer_null: 500/500 genes found: 100%|██████████| 1/1 [00:00<00:00, 11.38it/s]
Scoring Striatum_null: 434/434 genes found: 100%|██████████| 1/1 [00:00<00:00, 11.97it/s]


repeat 3 done
method                         EnrichMap (null)  AUCell  Scanpy  ssGSEA  Z score
region          gene set size                                                   
Pyramidal_layer 2                         1.445  46.928   0.544  21.820    0.003
                5                         1.463  47.066   0.595  21.696    0.004
                10                        1.476  46.899   0.568  21.976    0.007
                20                        1.507  46.888   0.601  22.237    0.010
                50                        1.480  46.920   0.594  21.168    0.015
                100                       1.478  46.911   0.597  22.404    0.018
                200                       1.553  46.945   0.640  22.589    0.008
                500                       1.733  46.917   0.602  21.759    0.016
Striatum        2                         1.407  46.992   0.549  22.322    0.001
                5                         1.438  46.823   0.572  22.050    0.005
              

In [11]:
# 5. GSVA: three interleaved repeats at the smallest and the largest set of each region (minutes per call)
rows = []
for rep in range(1, n_repeats + 1):
    for ct in regions:
        for n in [sizes[ct][0], sizes[ct][-1]]:
            t = runtime_time_method(adata, gene_expr, "GSVA", ct, gene_sets[ct][n], variants, enrichmap_variants, threads)
            rows.append({
                "repeat": rep,
                "region": ct,
                "method": "GSVA",
                "gene set size": n,
                "time (s)": t,
                "load average (1 min)": os.getloadavg()[0],
            })
            print(f"GSVA repeat {rep} {ct} n = {n}: {t:.1f} s", flush=True)
timings_gsva = pd.DataFrame(rows)

GSVA repeat 1 Pyramidal_layer n = 2: 468.2 s
GSVA repeat 1 Pyramidal_layer n = 500: 441.8 s
GSVA repeat 1 Striatum n = 2: 430.1 s
GSVA repeat 1 Striatum n = 434: 426.2 s
GSVA repeat 2 Pyramidal_layer n = 2: 424.0 s
GSVA repeat 2 Pyramidal_layer n = 500: 425.6 s
GSVA repeat 2 Striatum n = 2: 422.3 s
GSVA repeat 2 Striatum n = 434: 430.6 s
GSVA repeat 3 Pyramidal_layer n = 2: 422.8 s
GSVA repeat 3 Pyramidal_layer n = 500: 450.5 s
GSVA repeat 3 Striatum n = 2: 443.3 s
GSVA repeat 3 Striatum n = 434: 455.6 s


In [12]:
# 6. GSDensity: one R task per gene set and repeat, timed inside R (read from the cache); the plotted time is the
#    scoring on the in-memory matrix (MCA, kNN graph, random walk, labels), as for the Python methods
gsd_dir = working_dir + "data/gsd_cache/brain/SupFig5_6_7/runtime_20261029/"
rows = []
for rep in range(1, n_repeats + 1):
    for ct in regions:
        for n in sizes[ct]:
            with open(gsd_dir + f"rep{rep}_{ct}_top{n}/gsd_timing.json") as f:
                tim = json.load(f)
            rows.append({
                "repeat": rep,
                "region": ct,
                "method": "GSDensity",
                "gene set size": n,
                "time (s)": tim["total"] - tim["load"] - tim["spatial_kld"],
                "full pipeline incl. file loading and spatial test (s)": tim["total"],
                "load (s)": tim["load"],
                "MCA (s)": tim["mca"],
                "kNN graph (s)": tim["nn"],
                "random walk (s)": tim["rwr"],
                "spatial test (s)": tim["spatial_kld"],
            })
timings_gsd = pd.DataFrame(rows)

In [13]:
# 7. Mean, SD and range of the three repeats per method and size, and the published single run (recovered from the
#    original PDFs); the published EnrichMap run used the earlier default pipeline and is not shown
timings = pd.concat(
    [timings_py, timings_gsva, timings_gsd[["repeat", "region", "method", "gene set size", "time (s)"]]],
    ignore_index=True,
)
stats = (timings.groupby(["region", "method", "gene set size"])["time (s)"]
         .agg(mean="mean", sd="std", min="min", max="max", median="median", n_repeats="size").reset_index())

# the published single run (source_data/current/panels/SupFig7.pkl) is no longer on disk: no published overlay
published = pd.DataFrame({"region": pd.Series(dtype=str), "method": pd.Series(dtype=str),
                          "gene set size": pd.Series(dtype="int64"), "published time (s)": pd.Series(dtype=float)})

stats = stats.merge(published[["region", "method", "gene set size", "published time (s)"]],
                    on=["region", "method", "gene set size"], how="left")
stats["this server / published"] = stats["mean"] / stats["published time (s)"]
stats["method"] = pd.Categorical(stats["method"], methods, ordered=True)
stats = stats.sort_values(["region", "method", "gene set size"]).reset_index(drop=True)

In [ ]:
# 8. Extended Data Fig. 5d: runtime against gene set size (all sizes and n <= 50)
published_methods = ["AUCell", "GSVA", "ssGSEA", "Z score", "Scanpy"]

fig, axes = plt.subplots(1, 4, figsize=(11.0, 2.7), sharey=True)
for i, ct in enumerate(regions):
    for j, max_size in enumerate([None, 50]):
        ax = axes[2 * i + j]
        runtime_vs_size_axis(ax, stats, published, ct, sizes[ct], methods, published_methods, colours,
                             max_size=max_size, log_x=False)
        ax.set_title(ct.replace("_", " ") + (" (up to 50 genes)" if max_size else ""), fontsize=8)
        if 2 * i + j:
            ax.set_ylabel("")
runtime_method_legend(axes[-1], methods, colours)
plt.tight_layout()
set_box_ratio(fig)
plt.savefig(figure_dir + "ExtendedDataFig5d_runtime.pdf", bbox_inches="tight")
plt.show()
stats.assign(**{"shown in n <= 50 panel": stats["gene set size"] <= 50}).to_csv(
    source_dir + "ExtendedDataFig5d_runtime.csv", index=False
)